In [0]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import time

dbutils.widgets.text("bronze_catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

TABLE_NAME = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_history_weather_demo"

def generate_poland_grid():
    lat_min, lat_max = 49.0, 54.8
    lon_min, lon_max = 14.1, 24.2
    lat_step, lon_step = 0.30, 0.52
    
    grid = []
    point_id = 1
    for lat in np.arange(lat_min, lat_max, lat_step):
        for lon in np.arange(lon_min, lon_max, lon_step):
            grid.append({"id": f"pl_35km_{point_id}", "lat": round(lat, 4), "lon": round(lon, 4)})
            point_id += 1
    return grid

print("Формирование сетки и скачивание реальных данных с Open-Meteo...")
grid = generate_poland_grid()
all_rows = []
chunk_size = 30 # Запрашиваем по 30 координат за раз, чтобы не перегрузить API

for i in range(0, len(grid), chunk_size):
    chunk = grid[i:i+chunk_size]
    lats = ",".join([str(p['lat']) for p in chunk])
    lons = ",".join([str(p['lon']) for p in chunk])
    
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lats}&longitude={lons}&start_date=2021-02-08&end_date=2021-02-14&hourly=soil_temperature_0_to_7cm,precipitation"
    response = requests.get(url).json()
    
    if isinstance(response, list):
        for idx, loc_data in enumerate(response):
            grid_id = chunk[idx]['id']
            lat = chunk[idx]['lat']
            lon = chunk[idx]['lon']
            hourly = loc_data.get('hourly', {})
            
            if not hourly: continue
                
            for t, temp, precip in zip(hourly['time'], hourly['soil_temperature_0_to_7cm'], hourly['precipitation']):
                dt = pd.to_datetime(t)
                
                # Безопасное приведение типов (API иногда возвращает None для битых станций)
                temp = float(temp)
                precip = float(precip)
                
                all_rows.append({
                    "grid_id": grid_id,
                    "latitude": lat,
                    "longitude": lon,
                    "weather_time": dt, # Используем часовое время
                    "precipitation_mm": round(precip, 2),
                    "soil_temperature_c": temp,
                    "eventhub_enqueued_time": dt,
                    "ingest_timestamp": datetime.now()
                })
    time.sleep(1) # Пауза, чтобы API не заблокировал нас

pdf = pd.DataFrame(all_rows)
df_bronze = spark.createDataFrame(pdf)
df_bronze.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Bronze таблица {TABLE_NAME} создана. Скачано {df_bronze.count()} записей (интервал 1 час).")